## Подготовка окружения

In [1]:
import numpy as np
import torch
import torch.nn as nn
import re

from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.normalizers import Sequence, NFKC, Lowercase
from tokenizers.pre_tokenizers import Sequence as PreTokenSequence
from tokenizers.pre_tokenizers import Whitespace, Punctuation, Digits

from torch.nn.utils.rnn import pad_sequence

from torch.utils.data import DataLoader, TensorDataset

## Загрузка данных

In [2]:
with open("data/TinyStories-train.txt", "r", encoding="utf-8") as file:
    text = file.read()

train_corpus = text.split('<|endoftext|>')

In [3]:
train_corpus[:5]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.\n',
 '\nOnce upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were

In [4]:
with open("data/TinyStories-valid.txt", "r") as file:
    text = file.read()

valid_corpus = text.split('<|endoftext|>')

In [5]:
valid_corpus[:5]

[' Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."\nAfter playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.\n',
 '\nOnce upon a time, in a big forest, there lived a rhinoceros named Roxy. Roxy loved to climb. She climbed trees, rocks, and hills. One day, Roxy found an icy hill. She had never seen anything like it before. It was shiny and cold, and she wanted to climb it.\nRoxy tried to climb the icy hill, but it was very slippery. She tried again and again, but she kept falling down. Roxy was sad. She wanted to climb the icy hill so much. Then, she saw a little bird named Billy. Billy saw that Roxy was sad and asked, "Why are you sad, Roxy?"\nRoxy told Billy about the icy hill and how she couldn\'t climb it. Billy said, "I have an idea! Let\'s f

## Предварительная обработка

Чтобы обучать decoder-only Transformer на наших данных, нужно очистить текст, преобразовать в токены, а затем преобразовать токены в эмбеддинги.

### Очистка сырого текста

Для классификации и генерации текст чистят по-разному, потому что у моделей разные цели.

В классификации модель должна понять общий смысл текста и отнести его к какому-то классу: например, отзыв положительный или отрицательный, письмо спам или не спам, сообщение токсичное или нормальное. Поэтому там можно сильнее упрощать текст: убирать лишние символы, иногда пунктуацию, приводить слова к нижнему регистру, удалять стоп-слова. Главное — сохранить смысл.

А в генерации модель должна не просто понять смысл, а научиться писать текст. То есть она должна уметь ставить запятые, точки, вопросительные знаки, сохранять порядок слов, структуру предложений и стиль. Поэтому для decoder-only модели нельзя слишком сильно чистить текст. Если убрать пунктуацию, модель потом и сама будет генерировать текст без нормальных точек и запятых.

Не будем учить модель абзацам, поэтому, заменим `\n`, `\t`, красивые кавычки на пробельный символ.

In [6]:
def clean_text(text: str) -> str:
    text = text.replace("\n", " ")
    text = text.replace("\t", " ")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [7]:
train_corpus = [clean_text(doc) for doc in train_corpus]
valid_corpus = [clean_text(doc) for doc in valid_corpus]

In [8]:
train_corpus[:5]

['One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt. Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt." Together, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.',
 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong. One day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were fallin

## Токенизация

Токенизацию будем выполнять на уровне слов с помощью библиотеки `tokenizers` от Hugging Face. В процессе обучения токенизатора на обучающем корпусе сразу будет построен словарь токенов `token -> id`.

Обязательно сделаем специальные токены: `<bos> - Beginning Of Sequence`, `<eos> — End Of Sequence`, `<pad> — Padding`, `<unk> — Unknown`

Используем такой конвейер токенизации:

1) Normalization — `NFKC`, `Lowercase`

    NFKC — нормализует Unicode-символы.

    Lowercase — переводит текст в нижний регистр.

2) Pre-tokenization — `Whitespace`, `Punctuation`, `Digits`

    Whitespace — разбивает текст по пробелам и границам слов.

    Punctuation — отделяет знаки препинания от слов.

    Digits — отделяет числа от букв.

3) Model — `WordLevel`

    WordLevel — сопоставляет каждый токен с числовым индексом из словаря. Если токена нет в словаре, он заменяется на `<unk>`.

Возьмем размер словаря 20 000 самых частых токенов. Все остальные редкие токены, которые не попадут в словарь, будут заменяться специальным токеном `<unk>`.

In [9]:
tokenizer = Tokenizer(model=WordLevel(unk_token="<unk>"))
tokenizer.normalizer = Sequence([
    NFKC(), Lowercase()
    ])
tokenizer.pre_tokenizer = PreTokenSequence([
    Whitespace(), 
    Punctuation(), 
    Digits()
    ])
trainer = WordLevelTrainer(
    vocab_size=20000,
    special_tokens=["<pad>", "<unk>", "<bos>", "<eos>"]
)

tokenizer.train_from_iterator(train_corpus, trainer=trainer)  # pyright: ignore[reportUnknownMemberType]

Проверим работу токенизатора.

In [10]:
print(tokenizer.get_vocab_size()) # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
print(tokenizer.encode('One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp.').tokens) # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]
print(tokenizer.encode('One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp.').ids) # pyright: ignore[reportUnknownMemberType, reportUnknownArgumentType]

20000
['one', 'day', ',', 'a', 'little', 'girl', 'named', 'lily', 'found', 'a', 'needle', 'in', 'her', 'room', '.', 'she', 'knew', 'it', 'was', 'difficult', 'to', 'play', 'with', 'it', 'because', 'it', 'was', 'sharp', '.']
[35, 28, 7, 9, 39, 53, 77, 24, 112, 9, 1776, 21, 16, 188, 4, 13, 160, 14, 11, 1315, 8, 49, 23, 14, 192, 14, 11, 864, 4]


Выведем весь наш словарь на 20000 токенов.

In [11]:
vocab = tokenizer.get_vocab() # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
vocab

{'whiz': 12069,
 'shaylee': 19698,
 'splish': 8961,
 'chuffed': 19284,
 'skyscrapers': 15994,
 'tweezers': 11762,
 'peaked': 7840,
 'spun': 1366,
 'vibrate': 10511,
 'pillows': 2709,
 'unfortunately': 2519,
 'foundations': 19390,
 'babby': 16054,
 'disasters': 17557,
 'varoon': 19795,
 'cole': 7781,
 'amie': 7899,
 'whisk': 6968,
 'jenkins': 8052,
 'grease': 9912,
 'oval': 8286,
 'been': 322,
 'bullseye': 10731,
 'flower': 417,
 'jewelry': 2452,
 'carts': 6339,
 'chucked': 14665,
 'lyall': 16407,
 'dax': 11371,
 'acrobatic': 17158,
 'spirals': 10934,
 'scolded': 2402,
 'robots': 4190,
 'zachary': 19183,
 'tugged': 3111,
 'mad': 800,
 'tobi': 8939,
 'rebecca': 4663,
 'helma': 18405,
 'collage': 6733,
 'help': 90,
 'patsy': 10891,
 'scorching': 14907,
 'dentists': 18842,
 'waistcoat': 11704,
 'levels': 9494,
 'stupid': 1580,
 'biggy': 7322,
 'feeb': 17992,
 'madam': 11228,
 'kart': 10559,
 'countless': 10792,
 'descent': 13453,
 'coronation': 17220,
 'weekends': 8460,
 'd': 1070,
 'charl

Каждый документ в обучающем и валидационном корпусе преобразуем в последовательность индексов токенов. Для этого сначала применяем обученный токенизатор, затем добавляем специальные токены <bos> и <eos>. После этого все последовательности приводим к одной длине с помощью padding.

In [12]:
bos_id = tokenizer.token_to_id('<bos>') # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
eos_id = tokenizer.token_to_id('<eos>') # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
pad_id = tokenizer.token_to_id('<pad>')
print(pad_id, bos_id, eos_id) # pyright: ignore[reportUnknownArgumentType]

0 2 3


In [13]:
train_tokens = [[bos_id] + tokenizer.encode(doc).ids + [eos_id] for doc in train_corpus] # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]
valid_tokens = [[bos_id] + tokenizer.encode(doc).ids + [eos_id] for doc in valid_corpus] # pyright: ignore[reportUnknownMemberType, reportUnknownVariableType]

### Разбитие корпуса на X и y части

Задача состоит в генерации продолжения истории по её началу. Поэтому первая часть последовательности используется как исходный контекст для модели, а ошибка на этих токенах не считается. Loss рассчитывается только на второй части последовательности — на токенах продолжения, которые модель должна предсказать.

**Обучение именно на задачу "начало -> продолжение"**

Посчитаем длину каждой последовательности в токенах, чтобы определить реальную длину документов и правильно разделить их на исходный контекст и продолжение. Это также нужно для корректной работы с padding, так как `<pad>`-токены не должны учитываться при делении последовательности и расчёте ошибки.

`X` будет содержать всю последовательность токенов, кроме последнего токена. Первая часть последовательности будет использоваться как исходный контекст для модели.

`y` будет содержать ту же последовательность, но сдвинутую на один токен вправо. При этом ошибка будет считаться только на второй половине последовательности, то есть на продолжении истории. Для токенов из первой половины в `y` будет использоваться маска `<pad>`, чтобы они не участвовали в расчёте loss.

In [14]:
X_train = [tokens[:-1] for tokens in train_tokens]
y_train = [tokens[1:] for tokens in train_tokens]

In [15]:
X_valid = [tokens[:-1] for tokens in valid_tokens]
y_valid = [tokens[1:] for tokens in valid_tokens]

In [16]:
story_length_train = [len(doc) - 1 for doc in X_train]
story_length_valid = [len(doc) - 1 for doc in X_valid]

In [17]:
def create_masked_labels(labels, real_lengths):
    for i, tokens in enumerate(labels):
        prompt_len = real_lengths[i] // 2
        tokens[:prompt_len] = [pad_id] * prompt_len

    return labels

In [18]:
y_train = create_masked_labels(y_train, story_length_train)
y_valid = create_masked_labels(y_valid, story_length_valid)

### Сделаем паддинг последовательностей с помощью метода pad_sequence

Из-за того что у нас > 2 млн историй, то чтобы не было OOM (out of memory) мы будем очищать оперативную память

In [19]:
tensor_train_X = [torch.tensor(doc, dtype=torch.long) for doc in X_train] # pyright: ignore[reportUnknownVariableType]
padded_train_X = pad_sequence(sequences=tensor_train_X, batch_first=True, padding_value=vocab["<pad>"]) # pyright: ignore[reportUnknownArgumentType]

del tensor_train_X

In [20]:
tensor_train_y = [torch.tensor(doc, dtype=torch.long) for doc in y_train] # pyright: ignore[reportUnknownVariableType]
padded_train_y = pad_sequence(sequences=tensor_train_y, batch_first=True, padding_value=vocab["<pad>"]) # pyright: ignore[reportUnknownArgumentType]

del tensor_train_y

In [21]:
tensor_valid_X = [torch.tensor(doc, dtype=torch.long) for doc in X_valid] # pyright: ignore[reportUnknownVariableType]
padded_valid_X = pad_sequence(sequences=tensor_valid_X, batch_first=True, padding_value=vocab["<pad>"]) # pyright: ignore[reportUnknownArgumentType]

del tensor_valid_X

In [22]:
tensor_valid_y = [torch.tensor(doc, dtype=torch.long) for doc in y_valid] # pyright: ignore[reportUnknownVariableType]
padded_valid_y = pad_sequence(sequences=tensor_valid_y, batch_first=True, padding_value=vocab["<pad>"]) # pyright: ignore[reportUnknownArgumentType]

del tensor_valid_y

## Векторизация и подготовка данных к обучению

Сделаем обучаемые ембединги через реализацию слоя nn.embedings

In [23]:
embedding_layer = nn.Embedding(num_embeddings=len(vocab), embedding_dim=300, padding_idx=vocab["<pad>"])

Создадим TensorDataset и DataLoader

In [24]:
train_dataset = TensorDataset(padded_train_X, padded_train_y)
valid_dataset  = TensorDataset(padded_valid_X, padded_valid_y)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)

## Обучение модели

In [ ]:
class Transformer(nn.Module):
    def __init__(self, embedding_layer):
        super().__init__()

        self.embedding = embedding_layer

        
        

In [27]:
class PositionalEncoding():
    pass